In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
data = pd.read_csv("../data/telco_customer_churn.csv")

In [3]:
data.info()
data["Churn"] = data["Churn"].map({"Yes":1, "No":0})

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
from sklearn.model_selection import StratifiedShuffleSplit
split = StratifiedShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 42)
for train_index, test_index in split.split(data, data["Churn"]):
    train_set = data.loc[train_index]
    test_set = data.loc[test_index]

train_label = train_set["Churn"].copy()
test_label =  test_set["Churn"].copy()
train_nl = train_set.drop("Churn", axis = 1)
test_nl = test_set.drop("Churn", axis = 1)


In [5]:
train_set.info()

<class 'pandas.DataFrame'>
Index: 5634 entries, 3738 to 5639
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5634 non-null   str    
 1   gender            5634 non-null   str    
 2   SeniorCitizen     5634 non-null   int64  
 3   Partner           5634 non-null   str    
 4   Dependents        5634 non-null   str    
 5   tenure            5634 non-null   int64  
 6   PhoneService      5634 non-null   str    
 7   MultipleLines     5634 non-null   str    
 8   InternetService   5634 non-null   str    
 9   OnlineSecurity    5634 non-null   str    
 10  OnlineBackup      5634 non-null   str    
 11  DeviceProtection  5634 non-null   str    
 12  TechSupport       5634 non-null   str    
 13  StreamingTV       5634 non-null   str    
 14  StreamingMovies   5634 non-null   str    
 15  Contract          5634 non-null   str    
 16  PaperlessBilling  5634 non-null   str    
 17  PaymentM

In [6]:
test_set.info()

<class 'pandas.DataFrame'>
Index: 1409 entries, 437 to 5613
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        1409 non-null   str    
 1   gender            1409 non-null   str    
 2   SeniorCitizen     1409 non-null   int64  
 3   Partner           1409 non-null   str    
 4   Dependents        1409 non-null   str    
 5   tenure            1409 non-null   int64  
 6   PhoneService      1409 non-null   str    
 7   MultipleLines     1409 non-null   str    
 8   InternetService   1409 non-null   str    
 9   OnlineSecurity    1409 non-null   str    
 10  OnlineBackup      1409 non-null   str    
 11  DeviceProtection  1409 non-null   str    
 12  TechSupport       1409 non-null   str    
 13  StreamingTV       1409 non-null   str    
 14  StreamingMovies   1409 non-null   str    
 15  Contract          1409 non-null   str    
 16  PaperlessBilling  1409 non-null   str    
 17  PaymentMe

In [7]:
compare = pd.DataFrame({
    "FULL":data["Churn"].value_counts(normalize=True),
    "TRAIN":train_set["Churn"].value_counts(normalize=True),
    "TEST":test_set["Churn"].value_counts(normalize=True),
})
print(compare)

          FULL     TRAIN      TEST
Churn                             
0      0.73463  0.734647  0.734564
1      0.26537  0.265353  0.265436


In [8]:
print(train_set[train_set["InternetService"] == "No"]["DeviceProtection"].unique())
print(train_set[train_set["Dependents"] == "Yes"][["DeviceProtection", "OnlineSecurity", "OnlineBackup"]])

<StringArray>
['No internet service']
Length: 1, dtype: str
         DeviceProtection       OnlineSecurity         OnlineBackup
3151                   No                  Yes                   No
4860                   No                  Yes                  Yes
3810                   No                   No                   No
2666                  Yes                   No                  Yes
6950                   No                  Yes                   No
...                   ...                  ...                  ...
58    No internet service  No internet service  No internet service
608                   Yes                  Yes                  Yes
4332                   No                  Yes                   No
4635  No internet service  No internet service  No internet service
4546                  Yes                   No                  Yes

[1679 rows x 3 columns]


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [10]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

In [11]:
from sklearn.base import BaseEstimator, TransformerMixin

In [12]:
class StrToNum():
    def __init__(self):
        pass
    # fit method is needed so that the model can learn something from the data
    def fit(self, X, y = None):
        return self
    def transform(self, X, y = None):
        Y = X.copy()
        Y = Y.drop("customerID", axis = "columns")
        Y["TotalCharges"] = pd.to_numeric(Y["TotalCharges"], errors="coerce")
        Y = Y.replace("No phone service", "No" )
        Y = Y.replace("No internet service", "No")
        return Y

In [13]:
class C_OneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    # fit method is needed so that the model can learn something from the data
    def fit(self, X, y = None):
        return self

    def transform(self, X, y = None):
        Y = X.copy()
        for column in self.columns:
            Y = pd.get_dummies(Y, columns = [column], dtype=int)
            
            if column + "_No internet service" in Y:
                Y = Y.drop(column + "_No internet service", axis = "columns")
        return Y

In [14]:
class OrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns, values, multiple_values = False, dtype = int):
        self.columns = columns
        self.values = values
        self.multiple_values = multiple_values
        self.dtype = dtype
    def fit(self, X, y = None):
        self.fitted_ = True
        return self

    def transform(self, X, y = None):
        Y = X.copy()
        for i in range(len(self.columns)):
            
            if self.multiple_values:
                Y[self.columns[i]] = Y[self.columns[i]].map(self.values[i]).astype(self.dtype)
            else:
                mapped = Y[self.columns[i]].map(self.values)
                unmapped = Y[self.columns[i]][mapped.isna()]
                #print(unmapped)
                Y[self.columns[i]] = Y[self.columns[i]].map(self.values).astype(self.dtype)
        return Y

In [15]:
from scipy.stats import chi2_contingency

def chi_square(x,y, N, alpha):
    result = {}
    table = pd.crosstab(x, y)
    chi2_stat, p_value, _, _ = chi2_contingency(table)
    cramers_v = np.sqrt(chi2_stat/(N*(min(table.shape) - 1)))

    return cramers_v
    

In [16]:
class DropFeature(BaseEstimator, TransformerMixin):
    def __init__(self, columns, ignore, threshold):
        self.columns = columns
        self.ignore = ignore
        self.threshold = threshold

    def fit(self, X, y = None):
        Y = X.copy()
        
        self.result = []
        self.N = len(Y)
        
        self.all_columns_ = Y.select_dtypes(include = "int").columns
        
        self.to_drop_ = []
        for column in self.all_columns_:
            if column in self.ignore:
                continue
            cramers_v = chi_square(Y[column], train_label, self.N, 0.05)
            self.result.append({"column":column, "cramers_v":cramers_v})
            if cramers_v < self.threshold:
                self.to_drop_.append(column)
        self.fitted_ = True
        return self

    def transform(self, X, y = None):
        Y = X.copy()
        for column in self.to_drop_:
            if column in Y:
                Y = Y.drop(column, axis=1)
       
        #df = pd.DataFrame(self.result)
        #df = df.sort_values(by="cramers_v")
        #print(df.T.to_string())
        
        return Y
        

In [17]:
class AddFeature(BaseEstimator, TransformerMixin):
    def __init__(self, add_Loyalty, add_ExtraService, add_ChargePerService, add_EntertainmentService, add_ProtectionService, add_Vunerable):
        self.add_Loyalty = add_Loyalty
        self.add_ExtraService = add_ExtraService
        self.add_ChargePerService = add_ChargePerService
        self.add_EntertainmentService = add_EntertainmentService
        self.add_ProtectionService = add_ProtectionService
        self.add_Vunerable = add_Vunerable

    def fit(self, X, y = None):
        self.fitted_ = True
        return self

    def transform(self, X, y = None):
        Y = X.copy()
        Services = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                        "StreamingTV", "StreamingMovies", "PhoneService", 
                        "MultipleLines"]
        service_count = 0
        if self.add_Loyalty:
            Y["Loyalty"] = pd.cut(Y["tenure"], bins=[0, 15, 45, float('inf')], labels=[0, 1, 2], include_lowest = True).astype(int)
        if self.add_ExtraService:
            Y["ExtraService"] = Y[Services].sum(axis = 1) + (Y["InternetService_Fiber optic"]| Y["InternetService_DSL"])

        if self.add_ChargePerService:
            Y["ChargePerService"] = Y["MonthlyCharges"]/Y["ExtraService"]

        if self.add_EntertainmentService:
            Y["EntertainmentService"] = (Y["StreamingTV"] | Y["StreamingMovies"])
        if self.add_ProtectionService:
            Y["ProtectionService"] = Y["OnlineSecurity"] + Y["OnlineBackup"] + Y["DeviceProtection"]

        if self.add_Vunerable:
            Y["Vunerable"] = (Y["SeniorCitizen"] == 1) + (Y["Dependents"] == 0) + (Y["Partner"] == 0) + (Y["PaymentMethod_Electronic check"] == 1) + (Y["TechSupport"] == 0).astype(int)
       
        return Y

In [18]:
class MinMaxScaling(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
        
    def fit(self, X, y = None):
        self.fitted_ = True
        self.scaler_ = MinMaxScaler()
        self.scaler_.fit(X[self.columns])
        return self

    def transform(self, X, y = None):
        Y = X.copy()
        Y[self.columns] = self.scaler_.transform(Y[self.columns])

        return Y
    

In [19]:
class LogScale(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y = None):
        self.fitted_ = True
        return self

    def  transform(self, X, y = None):
        Y = X.copy()
        for column in self.columns:
            Y[column] = np.log10(Y[column])

        return Y

In [20]:
from xgboost import XGBClassifier

In [21]:
columns = ["PaymentMethod", "InternetService"]
drop_columns = [
    "StreamingTV_No", "StreamingMovies_No", "OnlineSecurity_No",
    "OnlineSecurity_Yes", "OnlineBackup_No", "DeviceProtection_No"]

pipeline = Pipeline([
    ("to_num", StrToNum()),
    ("one_hot", C_OneHotEncoder(columns)),
    ("ordinal1", OrdinalEncoder(["Contract", "gender"],[{"Month-to-month":0, "One year":1, "Two year":2}, {"Male":1, "Female":0}], True)),
    ("ordinal2", OrdinalEncoder(["Partner",
                                  "Dependents","PaperlessBilling",
                                  "PhoneService", "MultipleLines",
                                  "OnlineSecurity", "OnlineBackup", 
                                  "DeviceProtection", "TechSupport",
                                 "StreamingTV", "StreamingMovies"],
                                {"Yes":1, "No":0}, False)),
    ("new_feature", AddFeature(True, True, True, True, True, True)),
    ("drop", DropFeature(drop_columns, ["tenure"], 0.1)),
    ("xgb", XGBClassifier(random_state=42, eval_metric="auc"))
    #("log_scale", LogScale(["TotalCharges", "MonthlyCharges"])),
])

In [22]:
train_copy = train_nl.copy()

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import cross_val_score, StratifiedKFold, RandomizedSearchCV

In [24]:
neg, pos = (train_label).sum(), (train_label).sum()
val = neg/pos
print(val)

1.0


In [25]:
# ran_for = RandomForestClassifier(n_estimators = 100, max_depth = 5)
# knn = KNeighborsClassifier(n_neighbors = 5)
# model = XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.1,
#                       subsample=0.8, colsample_bytree=0.8, 
#                       scale_pos_weight=val,  # handles class imbalance
#                       random_state=42)

In [26]:
skf = StratifiedKFold(n_splits = 5, random_state= 42, shuffle = True)
# score1 = cross_val_score(model, new_train, train_label, scoring = "roc_auc", cv=skf)

# print(score1.mean())


In [27]:
parameter= [{
    'xgb__n_estimators': [5, 10, 15, 20],
    'xgb__max_depth':[3, 5, 8],
    'xgb__learning_rate':[0.1],
    'xgb__subsample':[0.3, 0.6, 0.8, 1],
    'xgb__colsample_bytree':[0.3, 0.6, 0.8, 1],
    #'xgb__scale_pos_weight':[],
    'xgb__reg_alpha':[0, 0.3, 0.6, 1],
    'xgb__reg_lambda':[1, 3, 5],

    'drop__threshold':[0.05, 0.08,  0.1, 0.12],
    'new_feature__add_Loyalty':[True, False],
    'new_feature__add_ExtraService':[True, False],
    'new_feature__add_ChargePerService':[True, False],
    'new_feature__add_EntertainmentService':[True, False],
    'new_feature__add_ProtectionService':[True, False],
    'new_feature__add_Vunerable':[True, False],
}, ]
gridSearch = RandomizedSearchCV(pipeline, n_iters = 200, parameter, cv = skf, scoring = "roc_auc", return_train_score = True)
gridSearch.fit(train_copy, train_label)

SyntaxError: positional argument follows keyword argument (803149468.py, line 19)